# Deliverables 1, 2 and 4 — Actual local circuit, raw-f extraction, and amplification

Constructs genuine Qiskit circuits for the full 90-dimensional local collision dilation for 1–10 coherent applications. Streaming is identity in this one-cell circuit; nontrivial global collision/streaming compilation remains a declared blocker.

In [1]:
from pathlib import Path
import sys
repo_root = Path.cwd().parent if Path.cwd().name == "deliverables" else Path.cwd()
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
output_dir = repo_root / "results" / "deliverables"
output_dir.mkdir(parents=True, exist_ok=True)

In [2]:
import json, numpy as np, pandas as pd
from quantum_aero.classical import LBMConfig
from quantum_aero.deliverables import initial_lattice_state
from quantum_aero.advanced import simulate_local_collision, amplitude_amplification_experiment, raw_state_observable_shots

field, omega, velocity_scale, dt = initial_lattice_state(LBMConfig(n=4, reynolds=100, t_end=.1, snapshots=2))
cell = field[0, 0]
circuit_rows = [simulate_local_collision(cell, omega, steps) for steps in (1, 2, 5, 10)]
circuit_df = pd.DataFrame(circuit_rows)
circuit_df

,steps,qubits,high_level_depth,success_probability,conditional_fidelity,expected_raw_attempts
0,1,8,2,6.320309e-03,1.0,1.582201e+02
1,2,9,3,3.989455e-05,1.0,2.506608e+04
2,5,12,6,1.005922e-11,1.0,9.941126e+10
3,10,17,11,1.010574e-22,1.0,9.895370e+21


In [3]:
aa_df = pd.DataFrame(amplitude_amplification_experiment(cell, omega, max_iterations=12))
best = aa_df.loc[aa_df.success_probability.idxmax()].to_dict()
print("Best exact amplitude-amplification point:", best)
aa_df

Best exact amplitude-amplification point: {'iterations': 9.0, 'success_probability': 0.9965590152075279, 'block_calls': 19.0}


,iterations,success_probability,block_calls
0,0,0.006320,1
1,1,0.055928,3
2,2,0.150159,5
3,3,0.279544,7
4,4,0.431081,9
5,5,0.589544,11
6,6,0.739009,13
7,7,0.864457,15
8,8,0.953282,17
9,9,0.996559,19


In [4]:
shot_df = pd.DataFrame([raw_state_observable_shots(field, shots, seed=7) for shots in (1_000, 10_000, 100_000, 1_000_000)])
shot_df

,shots,state_qubits,normalization,velocity_rmse_lattice,kinetic_energy_absolute_error,mode_11_absolute_error,unobserved_cells
0,1000,8,1.999757,0.136320,0.009332,0.936089,0
1,10000,8,1.999757,0.041711,0.000749,0.198220,0
2,100000,8,1.999757,0.012622,0.000036,0.047379,0
3,1000000,8,1.999757,0.003421,0.000001,0.006550,0


In [5]:
payload = {"omega": omega, "collision_circuits": circuit_rows, "amplitude_amplification": aa_df.to_dict("records"),
           "raw_f_observable_extraction": shot_df.to_dict("records"),
           "scope_warning": "Actual local Qiskit circuit. Global nontrivial streaming is not composed with the lifted collision."}
(output_dir / "07_actual_circuit_stateprep_amplification.json").write_text(json.dumps(payload, indent=2))
assert min(x["conditional_fidelity"] for x in circuit_rows) > 1-1e-12
assert best["success_probability"] > .95
print("PASS: exact local circuits, raw-f shot scaling, and amplification executed.")

PASS: exact local circuits, raw-f shot scaling, and amplification executed.
